# Hai thí nghiệm Future Work nêu ra

**A. Document order.** Vòng lặp chọn theo điểm, nên `P_comp` ra theo thứ tự liên
quan chứ không theo mạch văn gốc. Bài viết thẳng là chưa thử cách kia. Chạy lại
VimQA n=1003 với `--restore-order` để đo xem sắp lại theo `(doc_title, sent_id)`
có giúp reader nối hop dễ hơn không.

**B. Judge tách khỏi reader.** Bản gốc dùng chính reader làm answerability judge,
nên lần thử 0.5B đổi cả hai thứ một lúc - đó là confound mà Limitations thừa
nhận. Ghim judge ở 7B, chỉ hạ reader xuống 0.5B, thì mới tách được.

Repo public chưa có hai cờ này, nên ô dưới tự vá thay vì chờ push.


In [ ]:
# ══ CỬA CHẶN: GPU có chạy được bitsandbytes 4-bit không? ══
# Kaggle cấp NGẪU NHIÊN P100 (sm_60) hoặc T4 (sm_75) nếu metadata không ghi rõ
# machine_shape. PyTorch của Kaggle chỉ build cho sm_70 trở lên, nên trên P100
# bitsandbytes chết bằng SIGSEGV giữa lúc nạp trọng số:
#
#     Error named symbol not found at line 74 in file /src/csrc/ops.cu
#     rc=-11
#
# Lỗi đó mất ~2 phút mới hiện và thông báo không nói gì về nguyên nhân. Ô này
# phát hiện trong 5 giây và nói thẳng phải làm gì.
import torch

assert torch.cuda.is_available(), (
    'Không có GPU. Settings > Accelerator > GPU T4 x2, rồi chạy lại.')

name = torch.cuda.get_device_name(0)
major, minor = torch.cuda.get_device_capability(0)
gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU: {name}  sm_{major}{minor}  {gb:.1f} GB')

if major < 7:
    raise SystemExit(
        f'\n{name} là sm_{major}{minor} - PyTorch của Kaggle không hỗ trợ.\n'
        'Cần T4 (sm_75). Hai cách:\n'
        '  1. Settings > Accelerator > chọn "GPU T4 x2" (không phải "GPU P100")\n'
        '  2. Nếu đẩy bằng CLI: thêm "machine_shape": "NvidiaTeslaT4" vào\n'
        '     kernel-metadata.json\n'
        'Chạy tiếp trên P100 sẽ SIGSEGV lúc nạp mô hình 4-bit.')

print('✓ GPU chạy được bitsandbytes 4-bit')

In [ ]:
# ══ GHIM transformers VỀ 4.x - ĐÂY LÀ BẢN VÁ THẬT ══
# Kaggle nay ship transformers 5.0.0. llmlingua 0.2.2 viết cho 4.x, và hai bên
# đòi hai thứ LOẠI TRỪ NHAU cho cùng một object `past_key_values`:
#
#   transformers 5.0  ->  phải là Cache, gọi .get_seq_length()
#   llmlingua 0.2.2   ->  phải là list, lặp `for k, v in past_key_values`
#
# Không monkey-patch nào làm hài lòng cả hai, vì cùng một object đi qua cả hai
# nơi. SÁU lần vá đều chết vì cố làm điều bất khả. Đường LongLLMLingua đã được
# chạy thử THÀNH CÔNG trên 4.45.2 ở máy dev, nên ghim về đúng bản đó cũng khép
# luôn lỗ hổng "chạy được ở đây, chết trên Kaggle".
!pip install -q "transformers==4.45.2" llmlingua rank_bm25 tiktoken python-dotenv bitsandbytes accelerate 2>&1 | tail -3

import transformers
print('transformers =', transformers.__version__)
assert transformers.__version__.startswith('4.'), (
    f'Ghim KHÔNG ăn: đang là {transformers.__version__}. Nếu transformers đã '
    f'được import trước khi pip chạy thì phải Restart & Run All.')


In [ ]:
# Lấy mã nguồn bằng cách clone repo. Trước đây repo còn private nên notebook
# phải tải qua một gist công khai; repo public rồi, nên clone thẳng là nguồn duy
# nhất, không còn hai bản phải giữ đồng bộ.
import os, sys, subprocess, shutil

BASE = ('/content' if os.path.isdir('/content')
        else '/kaggle/working' if os.path.isdir('/kaggle/working')
        else os.getcwd())
REPO = os.path.join(BASE, 'ANLP-IterCOMP')
GIT_URL = 'https://github.com/nhantrnh/ANLP-IterCOMP.git'
# 'budget' nằm trong danh sách này từ một bản cũ; module đó không tồn tại ở
# repo nào, nên cửa chặn luôn báo thiếu và kernel chết trước khi chạy gì.
NEED = ['__init__', 'core', 'scorer', 'llm', 'reader', 'data', 'metrics', 'fertility', 'stats', 'stopping']

os.chdir(BASE)
shutil.rmtree(REPO, ignore_errors=True)
for m in list(sys.modules):
    if m == 'itercomp' or m.startswith('itercomp.'):
        del sys.modules[m]

subprocess.run(['git', 'clone', '--depth', '1', GIT_URL, REPO], check=True)

miss = [m for m in NEED
        if not os.path.exists(os.path.join(REPO, 'src', 'itercomp', m + '.py'))]
if miss:
    raise SystemExit('Bản clone thiếu module: ' + ', '.join(miss))

os.chdir(REPO)
if os.path.join(REPO, 'src') not in sys.path:
    sys.path.insert(0, os.path.join(REPO, 'src'))

sha = subprocess.run(['git', 'log', '-1', '--format=%h %s'],
                     capture_output=True, text=True).stdout.strip()
print('cwd:', os.getcwd(), '| src/scripts/:', os.path.isdir('src/scripts'))
print('đủ', len(NEED), 'module | commit:', sha)


In [ ]:
import urllib.request
FILES = {
 'vimqa/validation.parquet':    ('nguyenlab/vimqa', 'data/validation-00000-of-00001.parquet'),
 '2wiki/dev.parquet':           ('xanhho/2WikiMultihopQA', 'dev.parquet'),
 'hotpotqa/validation.parquet': ('hotpotqa/hotpot_qa', 'distractor/validation-00000-of-00001.parquet'),
 'musique/dev.jsonl':           ('dgslibisey/MuSiQue', 'musique_ans_v1.0_dev.jsonl'),
}
for dst, (repo, src) in FILES.items():
    p = 'data/' + dst
    os.makedirs(os.path.dirname(p), exist_ok=True)
    if not os.path.exists(p):
        urllib.request.urlretrieve(f'https://huggingface.co/datasets/{repo}/resolve/main/{src}', p)
    print(f'{dst:32s} {os.path.getsize(p)/1e6:6.1f} MB')

In [ ]:
# Vá tại chỗ: thêm --restore-order và --judge-model. Repo public còn bản cũ,
# và không có lý do gì phải public repo riêng chỉ để chạy một notebook.
import io

c = 'src/itercomp/core.py'
s = io.open(c, encoding='utf-8').read()
if 'restore_order' in s:
    print('core.py: đã có')
else:
    s = s.replace('    split_max_tokens: int | None = None,\n) -> IterCompResult:',
                  '    split_max_tokens: int | None = None,\n    restore_order: bool = False,\n) -> IterCompResult:')
    s = s.replace('        res.stopped_because = "hết vòng lặp"\n\n    return res',
                  '        res.stopped_because = "hết vòng lặp"\n\n'
                  '    if restore_order:\n'
                  '        res.selected.sort(key=lambda sg: (sg.doc_title, sg.sent_id))\n\n'
                  '    return res')
    io.open(c,'w',encoding='utf-8').write(s)
    assert 'restore_order' in io.open(c,encoding='utf-8').read(), 'vá core.py hỏng'
    print('core.py: đã vá')

r = 'src/scripts/run_eval.py'
s = io.open(r, encoding='utf-8').read()
if '--restore-order' in s:
    print('run_eval.py: đã có')
else:
    s = s.replace('                 k_max: float = AP_K_MAX):',
                  '                 k_max: float = AP_K_MAX, restore_order: bool = False):')
    s = s.replace('        self.adaptive_k, self.k_min, self.k_max = adaptive_k, k_min, k_max',
                  '        self.adaptive_k, self.k_min, self.k_max = adaptive_k, k_min, k_max\n'
                  '        self.restore_order = restore_order')
    s = s.replace('                       k_min=self.k_min, k_max=self.k_max)',
                  '                       k_min=self.k_min, k_max=self.k_max,\n'
                  '                       restore_order=self.restore_order)')
    s = s.replace('    ap.add_argument("--out", type=Path, default=None)',
                  '    ap.add_argument("--restore-order", action="store_true")\n'
                  '    ap.add_argument("--judge-model", default=None)\n'
                  '    ap.add_argument("--out", type=Path, default=None)')
    s = s.replace('                llm_model=args.reader_model, llm_4bit=args.load_4bit,\n'
                  '                adaptive_k=args.adaptive_k, k_min=args.k_min, k_max=args.k_max)',
                  '                llm_model=(args.judge_model or args.reader_model),\n'
                  '                llm_4bit=args.load_4bit,\n'
                  '                adaptive_k=args.adaptive_k, k_min=args.k_min, k_max=args.k_max,\n'
                  '                restore_order=args.restore_order)')
    io.open(r,'w',encoding='utf-8').write(s)
    t = io.open(r,encoding='utf-8').read()
    for probe in ['--restore-order','--judge-model','restore_order=args.restore_order','judge_model or args.reader_model']:
        assert probe in t, f'vá run_eval.py hỏng: thiếu {probe}'
    print('run_eval.py: đã vá')


In [ ]:
import subprocess, sys, os, time
R7='Qwen/Qwen2.5-7B-Instruct'; R05='Qwen/Qwen2.5-0.5B-Instruct'
os.makedirs('results', exist_ok=True)

def run(cmd, log):
    t0=time.time()
    with open(log,'w') as lf:
        p=subprocess.Popen(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        for line in p.stdout: print(line,end='',flush=True); lf.write(line)
        p.wait()
    print(f'\n[{time.time()-t0:.0f}s] rc={p.returncode}')
    return p.returncode

# A. document order, VimQA đầy đủ — so thẳng với vimqa_full_7b.json
A='results/vimqa_full_7b_ordered.json'
if os.path.exists(A): print('[bỏ qua]',A)
else: run([sys.executable,'-u','scripts/run_eval.py','--dataset','vimqa',
     '--limit','1003','--reader','hf','--reader-model',R7,'--load-4bit',
     '--itercomp-llm','hf','--scorer','dual','--lam','0.6','--max-iter','5',
     '--restore-order','--methods','itercomp','--out',A], 'A.log')

# B. judge ghim 7B, reader 0.5B — tách hai hiệu ứng mà lần 0.5B cũ trộn lẫn
B='results/hotpotqa_500_05b_judge7b.json'
if os.path.exists(B): print('[bỏ qua]',B)
else: run([sys.executable,'-u','scripts/run_eval.py','--dataset','hotpotqa',
     '--limit','500','--reader','hf','--reader-model',R05,
     '--itercomp-llm','hf','--judge-model',R7,'--load-4bit',
     '--scorer','dual','--lam','0.6','--max-iter','5',
     '--methods','raw,itercomp','--out',B], 'B.log')


In [ ]:
import shutil, os
BASE='/kaggle/working' if os.path.isdir('/kaggle/working') else os.getcwd()
z=shutil.make_archive(os.path.join(BASE,'results_ablation500'),'zip','results')
print('✓',z, f'({os.path.getsize(z)/1e6:.1f} MB)')